## Calculate relevant parameters for simulation

In [1]:
import numpy as np
import astropy.units as u
import astropy.constants as const

import h5py

## Initial parameters

In [2]:
env_params = {
    "ISM":{
        "n0":None,
        "rho0":2e-25,
        # "n0":1,
        # "rho0":None,
        "T0":1e4,
        "P0":None,
        "B":1e-6,
        "xmax":50,
    },
    "CMZ":{
        "n0":None,
        "rho0":1e-23,
        "T0":1e6,
        "P0":None,
        "B":1e-5,
        "xmax":30,
    },
    "X3A":{
        "n0":5e4,
        "rho0":None,
        "T0":7000,
        "P0":None,
        "B":0.0,
        "xmax":0.1,
    },
     "WR102":{
        "n0":40,
        "rho0":None,
        "T0":1e5,
        "P0":None,
        "B":1.2e-5,
        "xmax":15,
    },
}

In [9]:
# vstar for NRF
vstar = 100.0 * (u.km / u.s)
print(f"star velocity : {vstar:.3f}")

# mass of star
Mstar = 20 * u.M_sun
print(f"star mass [Msun]: {Mstar:.3f}")
print(f"star mass [g]: {Mstar.to(u.g):.6e}\n")

# set the environmental condition
env = "ISM"
env_param_dict = env_params[env]

star velocity : 100.000 km / s
star mass [Msun]: 20.000 solMass
star mass [g]: 3.976820e+34 g



In [10]:


# environment parameters

if env_param_dict["n0"] != None:
    rho0 = 0.61 * const.m_p.cgs * env_param_dict["n0"] * u.cm**-3
else:
    rho0 = env_param_dict["rho0"] * (u.g / u.cm**3)

n0 = (rho0 / (0.61 * const.m_p.cgs)).to(u.cm**-3)

T0 = env_param_dict["T0"]
P0 = env_param_dict["P0"]  # P0 / KB

adi_idx = 5/3

if T0 != None:
    T0 = T0 * u.K
    # calculate ambienmmt pressure from temperature and density
    cs = np.sqrt(adi_idx * const.k_B.cgs * T0 / (const.m_p.cgs))
    P0 = rho0 * cs**2 / adi_idx
elif P0 != None:
    P0 = (P0 * (u.K / u.cm**3)) * const.k_B.cgs  # convert back to P0 units
    # calculate temperature from pressure and densitu
    cs = np.sqrt(adi_idx * P0 / rho0)
    T0 = (cs**2 * const.m_p.cgs / (adi_idx * const.k_B.cgs)).to(u.K)

B = env_param_dict["B"] * (u.G)

print(f"density : {rho0:.3e}")
print(f"number density : {n0:.3f}")
print(f"temperature : {T0:.3e}")
print(f"Pressure : {P0:.3e}")
print(f"B : {B:.1e}")

print(f"sound speed : {cs.to(u.km/u.s):.3f}")

xmin = 0
xmax = env_param_dict["xmax"] * u.pc
print(f"Size of a single block: [{xmin:1f}, {xmax:.1f}] pc")

density : 2.000e-25 g / cm3
number density : 0.196 1 / cm3
temperature : 1.000e+04 K
Pressure : 1.651e-13 erg / cm3
B : 1.0e-06 G
sound speed : 11.729 km / s
Size of a single block: [0.000000, 50.0 pc] pc


### Calculate wind parameters from mass of star

In [11]:
assert np.min(Mstar) >= 9 * u.M_sun  # since the datafile only has from solarmass 9 and onwards

# read the data
with h5py.File("../../data/source_stellar.h5", "r") as f:

    times = f[str(int(Mstar.value))]["times"][()] * u.s
    vwinds = f[str(int(Mstar.value))]["vwinds"][()] * (u.cm / u.s)
    Mdots = f[str(int(Mstar.value))]["Mdots"][()] * (u.g / u.s)

    print(np.max(vwinds.to(u.km/u.s)), np.max(Mdots.to(u.M_sun / u.yr)))

    tlim_idx = np.digitize(1500, times.to_value(u.kyr), right=True)  # where the time-independence breaks down

    vwind = np.mean(vwinds[:tlim_idx])
    Mdot = np.mean(Mdots[:tlim_idx])

print(f"wind velocity: {vwind.to(u.km/u.s):.4f}")
print(f"mass loss rate: {Mdot.to(u.M_sun/u.yr):.4e}")

2800.6000000000004 km / s 0.00015358844524181902 solMass / yr
wind velocity: 2741.9644 km / s
mass loss rate: 2.5970e-08 solMass / yr


### Calculate the approximate domain size that we need

In [12]:

# check the approximate domain size we need
tmax = 500 * u.kyr
dmax = (tmax * (np.sqrt(vstar**2 + (0.5 * vwind)**2))).to(u.pc) 
print(f"Domain size that we need with tmax={tmax}: {dmax:.2f}")

Domain size that we need with tmax=500.0 kyr: 702.92 pc


### Characteristic length: stand-off radius

In [13]:
def stand_off_radius(Mdot, vw, rho0, vstar, P0, adi_idx, B):
    cism2 = (adi_idx * P0 / rho0).to(u.km**2/u.s**2)
    valf2 = (B**2 / (const.mu0.si * rho0)).to(u.km**2/u.s**2)

    return (np.sqrt((Mdot * vw) / (4 * np.pi * rho0 * (vstar**2 + cism2 + valf2)))).to(u.pc)


In [14]:
R0 = stand_off_radius(Mdot, vwind, rho0, vstar, P0, adi_idx, B)
print(f"stand-off radius: {R0:.5f}")

stand-off radius: 0.42922 pc


### Diffusion coefficent

In [9]:
kappa_par_10 = 1e25 * (u.cm**2 / u.s)
delta = 0.5
encr = 1 * (u.GeV)

kappa_par = kappa_par_10 * (encr / (10 * u.GeV))**delta
kappa_perp = kappa_par * 0.01
print(f"diffusion coefficient: parallel {kappa_par:.5e}, perp {kappa_perp:.5e}")

diffusion coefficient: parallel 3.16228e+24 cm2 / s, perp 3.16228e+22 cm2 / s


### Dynamical timescales & time resolution

Calculate from approximate lifetimie of object

In [10]:
tdyn = (R0 / vstar).to(u.kyr)
print(f"dynamical timescale: {tdyn:.5f}")

dynamical timescale: 42.99773 kyr


In [11]:
tdiff = (R0**2 / kappa_par_10).to(u.kyr)
print(f"diffusion timescale: {tdiff:.5f}")

diffusion timescale: 52.50946 kyr


## Grid size

- should be less than the stand-off radius
- also determines the wind injection radius = 2.5 cells * dx

In [15]:
max_amrlvl = 6 # as in fortran style

dx = (xmax - xmin) / (2**(max_amrlvl - 1) * 8)

print(f"grid resolution: {dx.to(u.pc):.6f}")

assert dx < R0  # assertion error if its larger than R0


rwind = 4 * dx
print(f"wind injection radius [pc]: {(rwind).to(u.pc):.6f}")
print(f"wind injection radius [cm]: {(rwind).to(u.cm):.6e}")

# also make sure that the wind velocity is smaller than the stand-off radius
assert 1.0 * rwind < R0

grid resolution: 0.195312 pc
wind injection radius [pc]: 0.781250 pc
wind injection radius [cm]: 2.410686e+18 cm


## Total number of cells per dimension

Calcualted by the AMR level and the number of blocks per direction

In [ ]:
# number of blocks in each direction
nblocks_x = 2
nblocks_y = 1
nblocks_z = 1

# refinement factor, based on AMR level
refine_factor = 2 ** (max_amrlvl - 1)

# number of cells per block, usually always 8
n_cells_per_block = 8

# number of cells in each direction
ncells_x = nblocks_x * refine_factor * n_cells_per_block
ncells_y = nblocks_y * refine_factor * n_cells_per_block
ncells_z = nblocks_z * refine_factor * n_cells_per_block

print(f"Number of cells in each direction: {ncells_x} x {ncells_y} x {ncells_z}")

Number of cells in each direction: 512 x 256 x 256


## Nodes required for simulation

can be calculated by the amr level, the number of blocks per processor, and the number of processors per node

In [ ]:
max_blocks_per_processor = 500   # set in ./setup
processors_per_node = 16

# number of blocks in each direction
nblocks_x = 2
nblocks_y = 1
nblocks_z = 1

nblocks_tot = nblocks_x * nblocks_y * nblocks_z

# refinement factor, based on AMR level
refine_factor = 2 ** (max_amrlvl - 1)

# max number of blocks required, mutiply refinement factor in each direction
max_nblocks_tot = nblocks_tot * refine_factor**3
print(max_nblocks_tot)

# minimum number of nodes required for simulation
min_nodes_required = (1 / processors_per_node) * (1 / max_blocks_per_processor) * max_nblocks_tot

print(f"Minimum number of nodes required for max amr level of {max_amrlvl}: {min_nodes_required}")

65536
Minimum number of nodes required for max amr level of 6: 8.192


## Temporal resolution

can be estimated from the grid resolution / star velocity, since it should be able to capture such effects.

In [ ]:
cfl = 0.75

dt = (dx / vwind).to(u.kyr) * cfl
print(f"time resolution [kyr]: {dt.to(u.kyr):.6f}")
print(f"time resolution [s]: {dt.to(u.s):.8e}")

assert dt < tdyn

dt_cr = (dx**2 / (kappa_par_10 + kappa_perp)).to(u.kyr) * cfl

assert dt_cr < tdiff

time resolution [kyr]: 0.052237 kyr
time resolution [s]: 1.64846613e+09 s


In [ ]:
dx_min = (dt * vstar).to(u.pc)

print(dx_min)

min_amr_lvl = 1 + np.log2((xmax - xmin) / (dx_min * 8))
print(min_amr_lvl)

0.0016026944692531882 pc
12.9291410518798


## Conversions needed for flash.par

In [ ]:
# pc -> cm
print(f"{(50 * u.pc).to(u.cm):.8e}")

# kyr -> s
print(f"{(10 * u.kyr).to(u.s):.8e}")

# cm -> pc
print(f"{(7.71419395e+19 * u.cm).to(u.pc):.8f}")

# s -> kyr
print(f"{(  1.131740078196872803E+12  * u.s).to(u.kyr):.8f}")

# kyr -> s
print(f"{( 500 * u.kyr).to(u.s):.8e}")

# Msun -> g
print(f"{( 20 * u.M_sun).to(u.g):.8e}")

1.54283879e+20 cm
3.15576000e+11 s
24.99999999 pc
35.86267898 kyr
1.57788000e+13 s
3.97681974e+34 g
